In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.preprocessing import MultiLabelBinarizer

df = pd.read_csv("/kaggle/input/datasets/walidnishat/gamesssssss/Games_Dataset_Clean.csv")

df["log_reviews"] = np.log1p(df["review count"])

df["metascore_norm"] = df["metascore"] / 100

df["success"] = df["log_reviews"] * df["metascore_norm"]

df["tags"] = df["tags"].fillna("").apply(lambda x: [i.strip() for i in x.split(",")])

mlb = MultiLabelBinarizer()
tags_encoded = mlb.fit_transform(df["tags"])
tags_df = pd.DataFrame(tags_encoded, columns=mlb.classes_)

df["platform"] = df["platform"].fillna("")

df["windows"] = df["platform"].apply(lambda x: 1 if "windows" in x else 0)
df["mac"] = df["platform"].apply(lambda x: 1 if "mac" in x else 0)
df["linux"] = df["platform"].apply(lambda x: 1 if "linux" in x else 0)

df["release date"] = pd.to_datetime(df["release date"], errors="coerce")

df["release year"] = df["release date"].dt.year
df["release month"] = df["release date"].dt.month

current_year = pd.Timestamp.now().year
df["game_age"] = current_year - df["release year"]

price_features = ["current price", "discount %"]

def top_n(column, n=25):
    top = df[column].value_counts().nlargest(n).index
    return df[column].apply(lambda x: x if x in top else "Other")

df["developer_clean"] = top_n("developer")
df["publisher_clean"] = top_n("publisher")

dev_pub_df = pd.get_dummies(df[["developer_clean", "publisher_clean"]], drop_first=True)

features = pd.concat([
    tags_df,
    df[price_features],
    df[["windows", "mac", "linux"]],
    df[["release year", "release month", "game_age"]],
    dev_pub_df
], axis=1)

features = features.fillna(0)

target = df["success"]


X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42
)

rf = RandomForestRegressor(random_state=42)

param_dist = {
    "n_estimators": [200, 300, 400],
    "max_depth": [10, 15, 20, 25],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 3],
    "max_features": ["sqrt", "log2"]
}

search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train)

best_model = search.best_estimator_

y_pred = best_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Random Forest Result")
print("MSE:", mse)
print("MAE:", mae)
print("R2 Score:", r2)

importances = pd.Series(best_model.feature_importances_, index=features.columns)
importances = importances.sort_values(ascending=False)

print("\nTop 15 Important Features:")
print(importances.head(15))

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Random Forest Result
MSE: 3.0527522954539887
MAE: 1.3460257257771615
R2 Score: 0.5086255707894988

Top 15 Important Features:
current price              0.109553
discount %                 0.097286
release year               0.080986
game_age                   0.079337
release month              0.053599
Steam Trading Cards        0.045516
Casual                     0.027210
Remote Play on Tablet      0.025479
Steam Achievements         0.020634
Full controller support    0.017119
Steam Cloud                0.017016
Action                     0.016721
Indie                      0.016423
Adventure                  0.015657
Remote Play on Phone       0.013569
dtype: float64


In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.preprocessing import MultiLabelBinarizer

from xgboost import XGBRegressor

df = pd.read_csv("/kaggle/input/datasets/walidnishat/gamesssssss/Games_Dataset_Clean.csv")


df["log_reviews"] = np.log1p(df["review count"])
df["metascore_norm"] = df["metascore"] / 100
df["success"] = df["log_reviews"] * df["metascore_norm"]

df["tags"] = df["tags"].fillna("").apply(lambda x: [i.strip() for i in x.split(",")])

mlb = MultiLabelBinarizer()
tags_encoded = mlb.fit_transform(df["tags"])
tags_df = pd.DataFrame(tags_encoded, columns=mlb.classes_)

df["platform"] = df["platform"].fillna("")

df["windows"] = df["platform"].apply(lambda x: 1 if "windows" in x else 0)
df["mac"] = df["platform"].apply(lambda x: 1 if "mac" in x else 0)
df["linux"] = df["platform"].apply(lambda x: 1 if "linux" in x else 0)

df["release date"] = pd.to_datetime(df["release date"], errors="coerce")

df["release year"] = df["release date"].dt.year
df["release month"] = df["release date"].dt.month

current_year = pd.Timestamp.now().year
df["game_age"] = current_year - df["release year"]

price_features = ["current price", "discount %"]


def top_n(column, n=25):
    top = df[column].value_counts().nlargest(n).index
    return df[column].apply(lambda x: x if x in top else "Other")

df["developer_clean"] = top_n("developer")
df["publisher_clean"] = top_n("publisher")

dev_pub_df = pd.get_dummies(df[["developer_clean", "publisher_clean"]], drop_first=True)


features = pd.concat([
    tags_df,
    df[price_features],
    df[["windows", "mac", "linux"]],
    df[["release year", "release month", "game_age"]],
    dev_pub_df
], axis=1)

features = features.fillna(0)

target = df["success"]

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42
)


xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

param_dist = {
    "n_estimators": [300, 500, 700],
    "max_depth": [4, 6, 8, 10],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
    "gamma": [0, 0.1, 0.3],
    "reg_alpha": [0, 0.1, 1],
    "reg_lambda": [1, 2, 5]
}

search = RandomizedSearchCV(
    xgb,
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train)

best_model = search.best_estimator_


y_pred = best_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("XGBoost Results")
print("MSE:", mse)
print("MAE:", mae)
print("R2 Score:", r2)

importances = pd.Series(best_model.feature_importances_, index=features.columns)
importances = importances.sort_values(ascending=False)

print("\nTop 15 Important Features (XGBoost):")
print(importances.head(15))

Fitting 3 folds for each of 20 candidates, totalling 60 fits
XGBoost Results
MSE: 2.9429536156813123
MAE: 1.3359271229446394
R2 Score: 0.5262988892836683

Top 15 Important Features (XGBoost):
Steam Trading Cards                       0.174541
discount %                                0.050614
Remote Play on Tablet                     0.046792
Online Co-op                              0.043557
Steam Achievements                        0.031310
publisher_clean_Kagura Games              0.030876
game_age                                  0.030196
Casual                                    0.022895
Multi-player                              0.021848
Surround Sound                            0.021397
release year                              0.020934
publisher_clean_BFG Entertainment         0.020797
developer_clean_Bitwave Games, Toaplan    0.020307
Remote Play on Phone                      0.020213
current price                             0.015312
dtype: float32
